In [1]:
# lib installations
!pip install -U \
  llama-index-core \
  llama-index-embeddings-huggingface \
  llama-index-vector-stores-faiss \
  llama-index-llms-ollama \
  faiss-cpu


# Approach 3: Local RAG Pipeline with LlamaIndex, FAISS, and Evaluation

This notebook is a starting point for building a fully local, open‑source RAG (Retrieval‑Augmented Generation) system that:

- Ingests and indexes Storied Data reports (e.g., `.sdp` / JSON / text reports).
- Uses local embeddings and a FAISS-based vector store for semantic retrieval.
- Uses a local LLM (e.g., via Ollama) to synthesize a combined report from multiple source documents.
- Provides hooks for RAG evaluation with tools like RAGAS and TruLens.

The goal is not to generate new SDP files, but to consume existing generated reports, retrieve their content, and combine them into a new, unified analytical report.



## Table of Contents

1. [Project Setup and Configuration](#1-Project-Setup-and-Configuration)  
2. [Data Ingestion: Storied Data Reports](#2-Data-Ingestion-Storied-Data-Reports)  
3. [Embedding and Vector Store (FAISS)](#3-Embedding-and-Vector-Store-FAISS)  
4. [LLM Integration (Local LLM via Ollama or Equivalent)](#4-LLM-Integration-Local-LLM-via-Ollama-or-Equivalent)  
5. [Building the RAG Query Engine](#5-Building-the-RAG-Query-Engine)  
6. [Combined Report Generation Workflow](#6-Combined-Report-Generation-Workflow)  
7. [RAG Evaluation Hooks (RAGAS, TruLens)](#7-RAG-Evaluation-Hooks-RAGAS-TruLens)  
8. [Next Steps and Extension Ideas](#8-Next-Steps-and-Extension-Ideas)



---
## 1. Project Setup and Configuration

This section defines the core configuration and environment dependencies.

We assume:
- You have a directory of Storied Data reports (e.g., `.sdp` JSON, JSON exports, or text reports).
- You want to keep everything local and open‑source, including embeddings and LLM inference.
- You have (or plan to have) a local LLM runtime such as Ollama running on your machine.


In [2]:
# 1.1 Optional: install dependencies (run once per environment)
# NOTE: These are example commands; uncomment and adapt as needed.
# !pip install llama-index-core llama-index-llms-ollama llama-index-vector-stores-faiss
# !pip install sentence-transformers faiss-cpu chromadb
# !pip install ragas trulens-eval

import os
from pathlib import Path

# Base directories (adapt these paths to your environment)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"          # Directory containing Storied Data reports
INDEX_DIR = PROJECT_ROOT / "index"        # Directory where FAISS index / artifacts will be stored
INDEX_DIR.mkdir(exist_ok=True, parents=True)

# Embedding and LLM configuration
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # or 'BAAI/bge-small-en-v1.5'
USE_FAISS = True  # if False, LlamaIndex can fall back to in-memory index

# LLM (local) configuration example: Ollama
LOCAL_LLM_MODEL = "llama3:8b"  # example for Ollama; change as needed
OLLAMA_ENDPOINT = "http://localhost:11434"  # default Ollama endpoint

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Index directory: {INDEX_DIR}")


Project root: /Users/atheeshkrishnan/AK/DEV/hawkdove/storied
Data directory: /Users/atheeshkrishnan/AK/DEV/hawkdove/storied/data
Index directory: /Users/atheeshkrishnan/AK/DEV/hawkdove/storied/index


In [3]:
import os

os.makedirs("data", exist_ok=True)
print("data folder created:", os.listdir())

data folder created: ['approach3_llamaindex_faiss_rag_start.ipynb', 'index_faiss', 'approach3_llamaindex_faiss_ragas_trulens.ipynb', 'index', 'venv', 'data', 'v2_approach3.ipynb', 't1.ipynb']


# 2. Data Ingestion: Storied Data Reports

In this project, Storied Data reports are the primary data source.  
They may be stored as:

- `.sdp` files (JSON-like structure with aggregated commentary fields), or  
- `.json` files, or  
- `.txt` / `.md` textual exports.

The ingestion goal is to:
1. Load all relevant reports from a directory.
2. Extract the commentary / analytical content (e.g., the `"Comments"` field).
3. Wrap the content into LlamaIndex `Document` objects with appropriate metadata (e.g., source name, date, report type).


In [4]:
# 2.1 Example: helper to load and normalize Storied Data reports
import json
from typing import List, Dict, Any

try:
    from llama_index.core import Document
except ImportError:
    Document = None  # placeholder if llama-index is not installed yet


def load_sdp_file(path: Path) -> List[Dict[str, Any]]:
    """
    Load a single .sdp JSON file and extract commentary.
    This is a generic template; adapt the extraction logic to your actual Storied Data schema.

    Expected pattern (example):
    - Top-level JSON with a 'columns' array.
    - One of the columns has title 'Comments' and a 'values' list.

    Returns a list of dicts, each with:
    - 'text': comment text
    - 'metadata': {'source': filename, ...}
    """
    data = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
    comments = []

    columns = data.get("columns", [])
    for col in columns:
        if (col.get("title") or "").strip().lower() == "comments" and isinstance(col.get("values"), list):
            for idx, val in enumerate(col["values"]):
                if not isinstance(val, str) or not val.strip():
                    continue
                comments.append({
                    "text": val.strip(),
                    "metadata": {
                        "source": path.name,
                        "comment_index": idx,
                        # Add more metadata if available, e.g., date, category, etc.
                    },
                })
            break
    return comments


def load_all_reports(data_dir: Path) -> List[Dict[str, Any]]:
    """
    Load all Storied Data reports from the given directory.
    This function can be extended to handle multiple formats (.sdp, .json, .txt, etc.).
    """
    all_entries: List[Dict[str, Any]] = []

    for path in sorted(data_dir.glob("**/*")):
        if path.suffix.lower() == ".sdp":
            entries = load_sdp_file(path)
        elif path.suffix.lower() in {".json"}:
            # Optionally handle generic JSON reports here
            try:
                data = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
                # TODO: define how to extract commentary from generic JSON
                entries = []
            except Exception:
                entries = []
        elif path.suffix.lower() in {".txt", ".md"}:
            text = path.read_text(encoding="utf-8", errors="ignore")
            entries = [{"text": text, "metadata": {"source": path.name}}]
        else:
            continue

        all_entries.extend(entries)

    print(f"Loaded {len(all_entries)} total commentary entries from {data_dir}.")
    return all_entries


# 2.2 Convert raw entries into LlamaIndex Documents
def build_documents(entries: List[Dict[str, Any]]) -> List["Document"]:
    if Document is None:
        raise ImportError("llama-index-core is not installed. Please install it to use Document objects.")
    docs: List[Document] = []
    for e in entries:
        docs.append(Document(text=e["text"], metadata=e.get("metadata", {})))
    return docs


# Example: only run after DATA_DIR is populated
if DATA_DIR.exists():
    raw_entries = load_all_reports(DATA_DIR)
    print(f"Example entry: {raw_entries[0] if raw_entries else 'N/A'}")
else:
    print("DATA_DIR does not exist yet; please create it and add Storied Data reports.")


Loaded 1417 total commentary entries from /Users/atheeshkrishnan/AK/DEV/hawkdove/storied/data.
Example entry: {'text': '2021 will get worse before it gets better', 'metadata': {'source': 'TSI Global Index.sdp', 'comment_index': 0}}


# 3. Embedding and Vector Store (in-memory)

This section builds the vector index used for semantic retrieval.

Core ideas:
- Use a local embedding model (e.g., SentenceTransformers) to embed each document chunk.

In [5]:
# ======================================================
# 3. Build LlamaIndex vector index (in-memory)
# ======================================================

from typing import Optional

try:
    from llama_index.core import VectorStoreIndex, Settings
    from llama_index.core.node_parser import SentenceSplitter
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
except ImportError as e:
    VectorStoreIndex = None
    Settings = None
    SentenceSplitter = None
    HuggingFaceEmbedding = None
    print("LlamaIndex core / HF embeddings not installed:", e)


def build_index_from_documents(documents) -> Optional["VectorStoreIndex"]:
    """
    Build a LlamaIndex VectorStoreIndex in memory.

    Steps:
    - Configure a HuggingFace embedding model (same model name as EMBEDDING_MODEL_NAME).
    - Register it in LlamaIndex global Settings.
    - Split documents into smaller nodes (chunks) with SentenceSplitter.
    - Build an in-memory VectorStoreIndex over those nodes.

    Returns:
        VectorStoreIndex or None if something fails.
    """
    if VectorStoreIndex is None or Settings is None:
        raise ImportError("llama-index-core and llama-index-embeddings-huggingface must be installed.")

    # 1) Set up embedding model
    embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL_NAME)
    Settings.embed_model = embed_model

    # 2) Chunk documents into nodes
    splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=128)
    nodes = splitter.get_nodes_from_documents(documents)
    print(f"Generated {len(nodes)} nodes from {len(documents)} documents.")

    # 3) Build in-memory index
    index = VectorStoreIndex(nodes)
    print("In-memory LlamaIndex VectorStoreIndex built successfully.")
    return index


# ------------------------------------------------------
# Build the index once, after raw_entries/documents exist
# ------------------------------------------------------
INDEX: Optional["VectorStoreIndex"] = None

if DATA_DIR.exists() and "raw_entries" in globals() and raw_entries:
    try:
        documents = build_documents(raw_entries)
        INDEX = build_index_from_documents(documents)
    except Exception as e:
        print("Index build skipped due to error:", e)
else:
    print("Index not built yet. Ensure DATA_DIR has data and raw_entries are loaded.")


/Users/atheeshkrishnan/AK/DEV/hawkdove/storied/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generated 1417 nodes from 1417 documents.
In-memory LlamaIndex VectorStoreIndex built successfully.


In [6]:
print("Embedding model:", Settings.embed_model)
print("Embedding dim:", Settings.embed_model._model.get_sentence_embedding_dimension())

Embedding model: model_name='sentence-transformers/all-MiniLM-L6-v2' embed_batch_size=10 callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x154a211d0> num_workers=None embeddings_cache=None max_length=256 normalize=True query_instruction=None text_instruction=None cache_folder=None show_progress_bar=False
Embedding dim: 384


## 3.x FAISS Incorporation

- Use FAISS as the underlying vector store for efficient similarity search.
- Wrap FAISS with LlamaIndex's vector store abstraction, so we can use LlamaIndex's query engines.

In [7]:
# ======================================================
# 3.X.1 FAISS prerequisites check (no changes to in-memory INDEX)
# ======================================================

import numpy as np

try:
    import faiss
    faiss_ok = True
except Exception as e:
    faiss_ok = False
    faiss_err = e

print("NumPy version:", np.__version__)
if faiss_ok:
    print("FAISS version:", getattr(faiss, "__version__", "unknown"))
else:
    print("FAISS import failed:", faiss_err)
    print(
        "\nIf you previously saw '_ARRAY_API not found', it is almost always NumPy 2.x vs FAISS wheel mismatch.\n"
        "Fix in your venv terminal:\n"
        "  pip uninstall -y faiss-cpu faiss-gpu faiss\n"
        "  pip install 'numpy<2.0'\n"
        "  pip install faiss-cpu==1.7.4\n"
    )

NumPy version: 2.3.5
FAISS version: 1.13.1


In [10]:
# ======================================================
# 3.x.2 Build + persist FAISS index (separate from LlamaIndex)
# ======================================================

from pathlib import Path
import json
import numpy as np

FAISS_DIR = Path("index_faiss")          # stays in your storied/ folder
FAISS_DIR.mkdir(parents=True, exist_ok=True)

FAISS_INDEX_PATH = FAISS_DIR / "faiss.index"   # binary
FAISS_META_PATH  = FAISS_DIR / "meta.json"     # text mapping

# Safety: require corpus to exist
if "documents" not in globals() or not documents:
    raise RuntimeError("documents not found. Run sections 1–2 first to build raw_entries/documents.")

# Build nodes exactly like your in-memory LlamaIndex section (same chunking)
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=128)
nodes = splitter.get_nodes_from_documents(documents)
print(f"[FAISS] Generated {len(nodes)} nodes from {len(documents)} documents.")

# Use sentence-transformers directly for embeddings (same model as before)
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [n.get_content(metadata_mode="none") for n in nodes]

# Embed all node texts
X = st_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,    # IMPORTANT: allows IndexFlatIP (cosine via dot product)
).astype("float32")

dim = X.shape[1]
print(f"[FAISS] Embedding dim = {dim}, matrix shape = {X.shape}")

# Build FAISS index (cosine similarity via inner product on normalized vectors)
import faiss
index = faiss.IndexFlatIP(dim)
index.add(X)

# Persist to disk
faiss.write_index(index, str(FAISS_INDEX_PATH))

# Persist metadata (row -> text + optional node id)
meta = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "normalize_embeddings": True,
    "faiss_index_type": "IndexFlatIP",
    "dim": dim,
    "count": len(texts),
    "rows": [
        {
            "row": i,
            "node_id": getattr(nodes[i], "node_id", None),
            "text": texts[i],
        }
        for i in range(len(texts))
    ],
}

with open(FAISS_META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False)

print(f"[FAISS] Wrote: {FAISS_INDEX_PATH}")
print(f"[FAISS] Wrote: {FAISS_META_PATH}")
print("[FAISS] Index build complete.")

[FAISS] Generated 1417 nodes from 1417 documents.


Batches: 100%|██████████| 45/45 [00:03<00:00, 12.68it/s]

[FAISS] Embedding dim = 384, matrix shape = (1417, 384)
[FAISS] Wrote: index_faiss/faiss.index
[FAISS] Wrote: index_faiss/meta.json
[FAISS] Index build complete.


In [11]:
# ======================================================
# 3.x.3 Reload FAISS index + search (sanity check)
# ======================================================

from pathlib import Path
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

FAISS_DIR = Path("index_faiss")
FAISS_INDEX_PATH = FAISS_DIR / "faiss.index"
FAISS_META_PATH  = FAISS_DIR / "meta.json"

if not FAISS_INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing FAISS index file: {FAISS_INDEX_PATH}")
if not FAISS_META_PATH.exists():
    raise FileNotFoundError(f"Missing FAISS meta file: {FAISS_META_PATH}")

index = faiss.read_index(str(FAISS_INDEX_PATH))

with open(FAISS_META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

rows = meta["rows"]
print(f"[FAISS] Reloaded index. dim={meta['dim']} count={meta['count']}")

st_model = SentenceTransformer(meta["embedding_model"])

def faiss_search(query: str, k: int = 5):
    q = st_model.encode([query], normalize_embeddings=meta["normalize_embeddings"], convert_to_numpy=True).astype("float32")
    scores, idxs = index.search(q, k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx < 0:
            continue
        hits.append({
            "row": int(idx),
            "score": float(score),
            "text": rows[idx]["text"],
        })
    return hits

# sanity test
hits = faiss_search("inflation outlook and policy risks", k=5)
for i, h in enumerate(hits, 1):
    print(f"\n--- Hit {i} (score={h['score']:.3f}) ---\n{h['text'][:400]}...")

[FAISS] Reloaded index. dim=384 count=1417

--- Hit 1 (score=0.605) ---
If central banks accept an inflation overshoot, they need to be very clear on how much and how long. Otherwise, the fashionably overshooting central bankers themselves become the main inflationary risk. - .This whole thing becoming highly politicised and the tool Reps will use in the Fall....

--- Hit 2 (score=0.598) ---
Inflation, mid-terms and political stasis mean...

--- Hit 3 (score=0.575) ---
Inflation ? - Laws of economics vs FDR 2.0. �Congress & White House flooded the economy w/ $1.9 trillion in new spending in March, after about $4 trillion in Covid relief in 2020....

--- Hit 4 (score=0.480) ---
Inflation turning out to be the talking poinat Thanksgiving family dinner tables. Gas and inflation an unholy mix for JB, Dems claim BIF marks the start of something new - Reps claim it�s a swansong. Reality is its business as usual in DC....

--- Hit 5 (score=0.477) ---
Risk is a toxic mix of rising food prices 

In [15]:
# ======================================================
# 3.X.4 FAISS Retriever (NO LlamaIndex QueryEngine, NO OpenAI)
# ======================================================

from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
import json
import numpy as np

FAISS_RETRIEVER_READY = False

# Directory where you persisted artifacts in 3.x.2
# (Set this to the same dir you used earlier; based on your outputs it's "index_faiss")
FAISS_DIR = Path("index_faiss")
FAISS_INDEX_PATH = FAISS_DIR / "faiss.index"
FAISS_META_PATH = FAISS_DIR / "meta.json"

# Will be populated on load
_FAISS_INDEX = None
_FAISS_META: Optional[List[Dict[str, Any]]] = None
_FAISS_DIM: Optional[int] = None


def _load_faiss_retriever(
    index_path: Path = FAISS_INDEX_PATH,
    meta_path: Path = FAISS_META_PATH,
) -> Tuple[Any, List[Dict[str, Any]]]:
    """
    Load FAISS index and meta.json, supporting multiple meta formats:
    - list[dict] aligned to rows 0..N-1  (preferred)
    - dict with a list under common keys: items / data / metas / metadata
    - dict keyed by integer-like strings: {"0": {...}, "1": {...}}
    """
    import faiss  # relies on faiss-cpu installed

    if not index_path.exists():
        raise FileNotFoundError(f"Missing FAISS index file: {index_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing meta.json file: {meta_path}")

    idx = faiss.read_index(str(index_path))

    with meta_path.open("r", encoding="utf-8") as f:
        raw = json.load(f)

    meta: Optional[List[Dict[str, Any]]] = None

    # Case 1: already a list
    if isinstance(raw, list):
        meta = raw

    # Case 2: dict wrapper with list under a known key
    elif isinstance(raw, dict):
        for key in ("items", "data", "metas", "metadata", "rows", "records"):
            if key in raw and isinstance(raw[key], list):
                meta = raw[key]
                break

        # Case 3: dict keyed by ids ("0", "1", ...)
        if meta is None:
            # If most keys look like ints, convert to list by sorted key
            keys = list(raw.keys())
            int_like = 0
            for k in keys[: min(len(keys), 50)]:
                try:
                    int(k)
                    int_like += 1
                except Exception:
                    pass
            if keys and int_like >= max(1, len(keys[: min(len(keys), 50)]) // 2):
                meta = [raw[str(i)] for i in sorted(int(k) for k in raw.keys())]

    if meta is None or not all(isinstance(x, dict) for x in meta):
        raise ValueError(
            "meta.json format not recognized. Expected list[dict], "
            "or dict containing a list under items/data/metas/metadata, "
            "or dict keyed by integer-like strings."
        )

    # Optional: sanity check alignment (length can be >= ntotal, but must cover [0..ntotal-1])
    if len(meta) < idx.ntotal:
        raise ValueError(
            f"meta.json has only {len(meta)} entries but FAISS index has {idx.ntotal} vectors. "
            "Meta must cover at least 0..ntotal-1."
        )

    return idx, meta


def _embed_query(text: str) -> np.ndarray:
    """
    Embed the query using the same embedding model you used for building FAISS.
    Assumes EMBED_MODEL (SentenceTransformer) exists in globals OR rebuilds it.
    Returns a float32 normalized vector of shape (1, dim).
    """
    global EMBED_MODEL

    # If your notebook already has a sentence-transformers model loaded, reuse it.
    if "EMBED_MODEL" not in globals() or EMBED_MODEL is None:
        from sentence_transformers import SentenceTransformer
        EMBED_MODEL = SentenceTransformer(EMBEDDING_MODEL_NAME)

    q = EMBED_MODEL.encode([text], normalize_embeddings=True)
    q = np.asarray(q, dtype="float32")
    return q


def faiss_search(query: str, top_k: int = 10) -> List[Dict[str, Any]]:
    """
    Pure FAISS similarity search.
    Returns a list of hits: {rank, score, idx, text, ...meta}
    Score is cosine similarity if you built an IP index over normalized vectors.
    """
    global _FAISS_INDEX, _FAISS_META, _FAISS_DIM, FAISS_RETRIEVER_READY

    if _FAISS_INDEX is None or _FAISS_META is None:
        _FAISS_INDEX, _FAISS_META = _load_faiss_retriever()
        _FAISS_DIM = _FAISS_INDEX.d
        FAISS_RETRIEVER_READY = True
        print(f"[FAISS] Retriever loaded. dim={_FAISS_DIM} count={_FAISS_INDEX.ntotal}")

    qv = _embed_query(query)

    if qv.shape[1] != _FAISS_DIM:
        raise ValueError(
            f"Query embedding dim {qv.shape[1]} does not match FAISS dim {_FAISS_DIM}. "
            f"Ensure EMBEDDING_MODEL_NAME matches what you used to build the index."
        )

    # Search returns distances/scores and indices
    scores, ids = _FAISS_INDEX.search(qv, top_k)

    hits: List[Dict[str, Any]] = []
    for rank, (score, idx) in enumerate(zip(scores[0].tolist(), ids[0].tolist()), start=1):
        if idx == -1:
            continue
        meta = _FAISS_META[idx] if idx < len(_FAISS_META) else {}
        text = meta.get("text", meta.get("chunk", meta.get("content", "")))

        hits.append(
            {
                "rank": rank,
                "score": float(score),
                "idx": int(idx),
                "text": text,
                **{k: v for k, v in meta.items() if k != "text"},
            }
        )

    return hits


# Quick sanity (optional)
if FAISS_INDEX_PATH.exists() and FAISS_META_PATH.exists():
    try:
        _ = faiss_search("inflation outlook", top_k=3)
        print("[FAISS] faiss_search() ready.")
    except Exception as e:
        print("[FAISS] Retriever init failed:", e)
else:
    print("[FAISS] Persisted files not found yet. Run 3.x.2 to create faiss.index and meta.json.")


[FAISS] Retriever loaded. dim=384 count=1417
[FAISS] faiss_search() ready.


In [ ]:
print(FAISS_QUERY_ENGINE.query("Generate a combined macro report across the corpus."))


---
## 4. LLM Integration (Local LLM via Ollama or Equivalent)

This section configures the local LLM used for generation.

We assume:
- You have a local LLM runtime such as Ollama running (e.g., `ollama run llama3:8b`).
- We use the LlamaIndex LLM wrapper to abstract the model backend.

You can swap the local backend later (e.g., vLLM, text-generation-webui) as long as it exposes an HTTP or Python API.


In [16]:
# ======================================================
# 4. Configure LLM (Ollama) and build a query helper
# ======================================================

TOP_K = 5  # number of similar chunks to retrieve per question

from typing import Optional

try:
    from llama_index.llms.ollama import Ollama
    from llama_index.core import Settings
except ImportError as e:
    Ollama = None
    Settings = None
    print("LlamaIndex Ollama / core modules not fully installed:", e)


def init_ollama_llm(model_name: str = "llama3:8b") -> Optional["Ollama"]:
    """
    Try to initialize a local LLM via Ollama.

    Requirements (when you run this on your own machine, not Colab):
      - Ollama installed: https://ollama.com/
      - ollama serve running
      - Model pulled, e.g.: `ollama pull llama3:8b`

    In Colab this will normally fail (no local Ollama daemon), but the error
    message will be explicit so you know what to do when running locally.
    """
    if Ollama is None or Settings is None:
        print("Ollama or LlamaIndex core not available; cannot configure local LLM.")
        return None

    try:
        llm = Ollama(
            model=model_name,
            request_timeout=120.0,  # generous timeout for local inference
        )
        # Register in global Settings so LlamaIndex knows which LLM to use
        Settings.llm = llm
        print(f"Ollama LLM configured with model: {model_name}")
        return llm
    except Exception as e:
        print(
            "Could not configure Ollama LLM.\n"
            "Make sure Ollama is installed and running locally, "
            "and that the model is pulled (e.g. `ollama pull llama3:8b`)."
        )
        print("Underlying error:", e)
        return None


LLM = init_ollama_llm(model_name="llama3:8b")


def answer(question: str, top_k: int = TOP_K) -> str:
    """
    High-level helper:
      - Uses the global INDEX built in Section 3.
      - Retrieves top_k most relevant nodes.
      - Asks the configured LLM (if available) to answer based on those nodes.

    If no LLM is configured, it will fall back to returning the top snippets
    so you still get *some* output.
    """
    if INDEX is None:
        raise RuntimeError("INDEX is not built yet. Run the indexing section first.")

    # Build a query engine; if LLM is registered in Settings.llm it will be used automatically.
    # We can also pass `llm=LLM` explicitly to be safe.
    query_engine = INDEX.as_query_engine(
        similarity_top_k=top_k,
        response_mode="compact",
        llm=LLM if LLM is not None else None,
    )

    if LLM is None:
        print("WARNING: No LLM configured; returning a synthesized text-only response from retrieved nodes.")
    response = query_engine.query(question)
    return str(response)


Ollama LLM configured with model: llama3:8b


In [17]:
# 4.1 Configure LLM in LlamaIndex (NO ServiceContext)
try:
    from llama_index.llms.ollama import Ollama
    from llama_index.core import Settings
except ImportError:
    Ollama = None
    Settings = None

LLM = None

if Ollama is None or Settings is None:
    print("Ollama / LlamaIndex not available. Install llama-index-llms-ollama.")
else:
    try:
        # LOCAL_LLM_MODEL and OLLAMA_ENDPOINT should already be defined earlier
        LLM = Ollama(model=LOCAL_LLM_MODEL, base_url=OLLAMA_ENDPOINT, request_timeout=120.0)
        Settings.llm = LLM  # register globally
        print(f"Ollama LLM configured with model: {LOCAL_LLM_MODEL} @ {OLLAMA_ENDPOINT}")
    except Exception as e:
        print("Failed to configure Ollama LLM:", e)
        LLM = None

Ollama LLM configured with model: llama3:8b @ http://localhost:11434



---
## 5. Building the RAG Query Engine

With the index and LLM configured, we now build a RAG query engine that:

1. Takes a natural language query.
2. Retrieves the top‑K most relevant nodes from the vector index.
3. Feeds them, along with the query, into the local LLM.
4. Produces a synthesized response, optionally with references to source documents.


In [18]:
# 5.1 Construct LlamaIndex query engine (NO ServiceContext)
from typing import Any

QUERY_ENGINE: Any = None

if INDEX is not None:
    try:
        QUERY_ENGINE = INDEX.as_query_engine(
            similarity_top_k=10,
            response_mode="tree_summarize",
            llm=LLM  # optional; can be omitted if Settings.llm is set
        )
        print("Query engine initialized.")
    except Exception as e:
        print("Query engine initialization failed:", e)
else:
    print("Query engine not initialized. Ensure INDEX is built.")


# Example (when everything is configured):
# resp = ask("Summarize the key macroeconomic themes across all reports.")
# print(resp)


Query engine initialized.


### Sanity Checks

In [19]:
print(QUERY_ENGINE.query("In one sentence, what is this corpus about?"))

This corpus appears to be a collection of news updates and commentary on global events, focusing on issues such as political turmoil, economic crises, and shifts in power dynamics, particularly in the context of international relations and geopolitics.


In [20]:
retriever = INDEX.as_retriever(similarity_top_k=3)
hits = retriever.retrieve("What are the main themes?")
for i, h in enumerate(hits, 1):
    print(f"\n--- Hit {i} (score={getattr(h,'score',None)}) ---\n")
    print(h.text[:400])


--- Hit 1 (score=0.25976375993940803) ---

S strength, re-emergence of C-19 fears, collapse in lucratice Christmas tourism season bookings, poltiical unrest. EM's remain calm on the surface but are turbulent beneath. Old animostities being stirred up in N Africe.

--- Hit 2 (score=0.22678940017857555) ---

Russian pressure campaign against the West. There are three elements: -manufacturing a (potential) gas crisis-instrumentalising (if not manufacturing) a border crisis between Belarus and the EU raising the military temperature against Ukraine

--- Hit 3 (score=0.21813417851479508) ---

A�consistent theme�in the US approach to the Japan�South Korea row is to conduct diplomacy with Seoul and Tokyo behind closed doors so as not to appear as favouring one over the other.



---
## 6. Combined Report Generation Workflow

The core use case is to take multiple Storied Data reports and generate a new combined report that:

- Synthesizes themes across all documents.
- Highlights key macroeconomic narratives (inflation, labor market, growth, policy stance, risks).
- Optionally uses a fixed structure (sections, bullet points) suitable for downstream consumption.


In [21]:
# 6.1 Example: generate a combined report

COMBINED_REPORT_PROMPT = """
You are a macroeconomic research analyst.

You are given retrieved excerpts from multiple Storied Data reports
that discuss financial markets, macro conditions, and policy themes.

Task:
- Produce a single, unified report that synthesizes the main themes.
- Organize the report into sections:
  1) Executive Summary
  2) Inflation
  3) Labor Market
  4) Growth and Activity
  5) Financial Conditions and Markets
  6) Policy Outlook and Risks

Guidelines:
- Be concise but substantive (2–5 bullet points per section).
- Use only information grounded in the retrieved context.
- Avoid speculation or external knowledge.
- Do not mention that you are an AI model.
"""


def generate_combined_report() -> Any:
    if QUERY_ENGINE is None:
        raise RuntimeError("QUERY_ENGINE is not initialized.")
    query = (
        COMBINED_REPORT_PROMPT
        + "\n\nQuestion: Generate the combined macroeconomic report now, based on all available context."
    )
    response = QUERY_ENGINE.query(query)
    return response


In [22]:
# Example:
combined_resp = generate_combined_report()
print(combined_resp)

**Executive Summary**

The US economy is experiencing a complex mix of trends, with growth and activity slowing down while inflation remains elevated. The recent massive stimulus package has contributed to concerns about economic bubbles in certain markets. Meanwhile, the labor market continues to show signs of improvement, but millions remain unemployed.

**Inflation**

* Inflationary pressures persist, driven by strong post-pandemic recovery and government spending.
* Faster inflation is expected to linger well into next year.
* Real estate prices are soaring, and household debt in South Korea is worsening, creating concerns about an economic bubble similar to Japan's late 1980s situation.

**Labor Market**

* The US economy is growing at its slowest pace since the recovery began.
* Millions remain unemployed even as small businesses struggle to hire workers.
* Job market dynamics are mixed, with some signs of improvement but still significant challenges.

**Growth and Activity**

* 


---
## 7. RAG Evaluation Hooks (RAGAS, TruLens)

To move beyond “it feels right”, we can introduce evaluation tooling:

- RAGAS: Framework for evaluating RAG pipelines on metrics like:
  - Faithfulness (is the answer grounded in context?)
  - Context relevance
  - Answer semantic similarity to ground truth
- TruLens: Observability and evaluation toolkit for LLM systems, including:
  - Tracing which context chunks were used
  - Measuring hallucination risk
  - Inspecting prompt/response pairs

In this notebook, we provide scaffolding to integrate these tools later.


In [ ]:
# 7.1 RAGAS scaffolding (to be filled in when ground-truth data is available)

# NOTE: This is a template for future work.
# You will need a dataset of (question, ground_truth_answer, retrieved_context) pairs
# to properly run RAGAS metrics.

# from datasets import Dataset
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_relevancy

# # Example placeholder structure:
# eval_data = {
#     "question": [
#         "What are the main inflation themes across reports?",
#     ],
#     "answer": [
#         "<MODEL_GENERATED_ANSWER_HERE>",
#     ],
#     "contexts": [
#         ["<RETRIEVED_CONTEXT_CHUNK_1>", "<RETRIEVED_CONTEXT_CHUNK_2>"]
#     ],
#     "ground_truth": [
#         "<CURATED_GROUND_TRUTH_SUMMARY_HERE>",
#     ],
# }

# dataset = Dataset.from_dict(eval_data)
# result = evaluate(
#     dataset,
#     metrics=[faithfulness, answer_relevancy, context_relevancy],
# )
# print(result)


# 7.2 TruLens scaffolding (for tracing and observability)

# from trulens_eval import Tru
# from trulens_eval.tru_llamaindex import TruLlamaIndexApp

# tru = Tru()

# if INDEX is not None and QUERY_ENGINE is not None:
#     app = TruLlamaIndexApp(
#         query_engine=QUERY_ENGINE,
#         app_id="storieddata_rag_v1",
#     )
#     tru.run_dashboard()  # Optional: start dashboard to inspect traces



---
## 8. Next Steps and Extension Ideas

This notebook is a starting point. Possible next steps:

1. Improve ingestion  
   - Fine-tune the `.sdp` parsing logic (dates, categories, risk flags).
   - Add support for more formats (PDF → text, HTML pages, etc.).

2. Refine retrieval  
   - Experiment with different embedding models (e.g., `bge-small`, `gte-large`).  
   - Tune `chunk_size`, `chunk_overlap`, and `similarity_top_k`.

3. Enhance report generation  
   - Enforce stricter structure or JSON output for downstream systems.  
   - Add multiple “modes”: high-level summary, risk-only report, theme comparison across time.

4. Introduce proper evaluation  
   - Curate a small labeled set of Q&A pairs from your reports.  
   - Use RAGAS to get repeatable metrics.  
   - Use TruLens to inspect which chunks drive the model’s reasoning.

5. Modularize into a package  
   - Extract ingestion, indexing, query, and evaluation into separate modules.  
   - Add CLI or simple web UI (e.g., Streamlit, Gradio) for non-technical users.

From here, you can edit each section case by case to align with your exact data schema, local LLM setup, and evaluation needs.
